# 14c · 把一个分数拆成两个能量

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F：一条线，从一个分数到两个能量]
                 ├─ 14a 把稳定性变成能量 ✅
                 ├─ 14b 把测序读数变成两种测量 ✅
                 ├─ 14c 把一个分数拆成两个能量  ← 你在这里
                 ├─ 14d 把两个能量变成一张位点地图 ✅
                 ├─ 14e 把地图用到一个真实的药上 ✅
                 └─ 14f 把整条线讲出来 ✅

附录（不在这条线上）
                 └─ 14x 实验室该测多少个 ✅
```

## 这一章做什么

之前的模型对每个突变输出**一个分数**，表示破坏有多大。这一章改成输出**两个数**：

- **ΔΔG_fold**：突变对蛋白**稳定性**的影响（还能不能保持折叠）
- **ΔΔG_func**：突变对蛋白**功能**的影响（还能不能结合、催化）

单位都是 kcal/mol。

## 背景

**蛋白折叠。** 蛋白是一条氨基酸链，要折叠成特定的三维结构才能工作。没折叠的蛋白会被细胞降解。

**突变的两种破坏方式：**
- **破坏稳定性**：蛋白折不好，被降解，细胞里的蛋白量下降
- **破坏功能**：蛋白正常折叠，量不变，但负责结合或催化的部位坏了

两种情况的处理方式不同。稳定性问题可以用小分子把蛋白稳住（14e 的药理伴侣），功能问题不行。一个分数区分不了这两种情况。

**能量是什么。** ΔG_fold 表示蛋白处于折叠状态有多稳定，数值越大越稳定。它决定了细胞里有多少比例的蛋白是折叠的：
- ΔG_fold = 0 时，一半折叠
- ΔG_fold 每增加 1.4 kcal/mol，折叠与未折叠的数量比增大 10 倍
- 典型的蛋白约 +2 kcal/mol，即 97% 折叠

ΔG_func 同理，表示折叠后的蛋白处于工作状态（结合 / 催化）有多稳定。

ΔG 是蛋白本身（野生型）的值，**ΔΔG 是突变造成的变化**，也就是模型要预测的量。

**实验测的是什么。** 深度突变扫描（DMS）一次测一个蛋白的几千个突变，读数分三类：

| 实验类型 | 例子 | 读数反映什么 |
|---|---|---|
| 测蛋白量 | VAMP-seq、AbundancePCA、Domainome | **只反映稳定性** |
| 测活性 / 结合 | BindingPCA、酶活 | **稳定性和功能两者** |
| 直接测稳定性 | 14a 的 Tsuboyama | 直接就是 ΔΔG_fold（kcal/mol） |

**难点。** 大多数蛋白只有活性数据。活性低，可能是稳定性坏了，也可能是功能坏了，单靠活性数据分不开。

## 步骤

| 步 | 内容 |
|---|---|
| 1 | 换算器：把两个能量换算成实验读数，用于训练 |
| 2 | 什么时候能拆：同一个蛋白要同时有蛋白量和活性数据 |
| 3 | 训练 |
| 4 | 测试：KRAS，和实验测出的 7 组能量比较；按位置检验显著性；再加 CYP2C9、G6PD 两个蛋白 |
| 5 | 能量可加：用单突变预测双突变 |
| 6 | 记录 |

---

## 准备

加载工具、连接 Databricks、设定文件夹。

In [ ]:
import os, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from scipy import stats
from databricks.sdk import WorkspaceClient
from qafi_platform import tracking
from qafi_platform.lakehouse import sql
from qafi_core import dms, energy, stability, validation
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
WORK, MODELS = ROOT / "data/stability", ROOT / "data/models"   # 14a 的数据、07–09 的缓存
KRAS = ROOT / "data/kras"
KRAS.mkdir(parents=True, exist_ok=True)
print("训练用的处理器:", stability.device())

---

## 第 1 步 · 换算器（物理读数层）

### 在干什么
定义模型的最后一层：把网络输出的两个能量，换算成实验应该测到的读数。

### 为什么需要它
训练数据的标签是**实验读数，不是能量**。训练时：

1. 网络输出两个能量
2. 换算器把两个能量算成预测读数
3. 预测读数和真实读数算误差
4. 用误差更新网络

没有换算器，网络输出的能量就没有标签可以对，无法训练。

训练完成后，预测时直接取网络输出的两个能量，不再需要换算器。

### 怎么算
1. 突变后的能量 = 野生型能量 + 网络预测的 ΔΔG
2. 能量换成比例（玻尔兹曼分布），RT ≈ 0.6 kcal/mol（30 °C）：
   ```
   折叠的比例       = sigmoid(ΔG_fold / RT)
   折叠且工作的比例 = 1 / (1 + exp(−ΔG_func/RT) · (1 + exp(−ΔG_fold/RT)))
   ```
3. 比例换成该实验的读数：每组实验一个线性换算（scale、offset）

除了网络本身，模型还要学：每个蛋白的野生型折叠能、每组实验的野生型功能能和 scale / offset。

### 关键：不同实验用不同公式
- **测蛋白量**：只用折叠能
- **测活性**：两个能量都用
- **直接测稳定性**：直接等于 ΔΔG_fold

结果是：蛋白量数据只更新折叠能；活性数据里折叠能解释不了的部分，只能由功能能来解释。两个输出因此分别对应"稳定性"和"功能"。**这个分工由实验类型决定，不是由网络结构决定。**

### 用了什么技术，为什么
这套热力学模型来自 Lehner 实验室（Faure 等 2022，工具 MoCHI），是这个领域的标准做法。原方法逐个蛋白拟合，只能用在测过的蛋白上。这里能量由神经网络从序列预测，所以可以用在没测过的蛋白上。

它取代了 08a 的做法（每组实验各自标准化）：用物理模型代替统计处理。

### 执行
把一组能量代入公式，检查输出是否符合预期。这一格不训练，也不产生结果。

In [ ]:
g = np.linspace(-4, 4, 200)
table = pd.DataFrame({
    "ΔG_fold (kcal/mol)": g,
    "折好的比例": energy.fraction_folded(g),
    "折好且工作（功能能 +1）": energy.fraction_functional(g, 1.0),
    "折好且工作（功能能 −1）": energy.fraction_functional(g, -1.0),
})
print(table.iloc[::25].round(3).to_string(index=False))
print(f"\nRT = {energy.RT} kcal/mol；野生型典型值 2 kcal/mol 时折好的比例 = {energy.fraction_folded(2.0):.3f}")
print(f"一个 −3 kcal/mol 的突变之后 = {energy.fraction_folded(-1.0):.3f}")

### 结果怎么看
- 折叠能 0 → 折叠比例 0.5
- 折叠能 +2 → 0.97；−1 → 0.16
- 折叠能很低（−4）时，折叠且工作的比例接近 0：不折叠就不能工作
- 功能能从 +1 降到 −1，折叠且工作的比例整体下降，即使折叠能不变
- 折叠能偏低时，"折叠且工作（+1）"高于"折叠的比例"：和伙伴结合会让蛋白更稳定，这是三态模型的正常结果

这些性质写在 `tests/unit/test_energy.py` 里。

### 面试怎么说
> "The network predicts two free energies and a thermodynamic layer turns them into each assay's readout: abundance assays see the folded fraction, activity assays see folded-and-functional. The separation is imposed by the assay type, not by the architecture."

---

## 第 2 步 · 什么时候能拆

### 在干什么
确定在什么数据条件下，两个能量可以被区分开。这决定第 3 步用哪些数据训练。

### 背景：不可辨识
活性读数同时取决于折叠能和功能能。只有活性数据时，不同的（折叠能，功能能）组合可以给出相同的读数，模型无法确定是哪一种。这叫**不可辨识**（not identifiable）：一个方程，两个未知数。这是数据本身缺信息，调参数解决不了。

试过两种办法，都无效：
1. 对功能能加 L1 惩罚：功能能变成 0（相关 0.001），全部归给折叠能，总分不变
2. 活性数据不更新折叠能（MoCHI 的两步拟合）：两个能量的相关只从 0.82 降到 0.79

### 什么时候可以
**同一个蛋白同时有蛋白量和活性数据。** 蛋白量数据确定折叠能，活性数据剩下的部分确定功能能。两个方程，两个未知数。Lehner 实验室的 ddPCA 就是这种设计，14b 生成的也是这种成对数据。

### 执行
统计训练数据中，同时有两类实验的蛋白有多少。

In [ ]:
titles = sql(w, f"SELECT dataset_id, title FROM {CATALOG}.gold.dim_label_dataset").set_index("dataset_id").title
labels = pd.read_parquet(MODELS / "train_variants.parquet")
labels["reads"] = [energy.readout_kind(k, titles.get(d))
                   for k, d in zip(labels.data_kind, labels.dataset_id, strict=True)]
kinds = labels.groupby("uniprot_id").reads.apply(set)
paired = sorted(p for p, k in kinds.items() if {energy.ABUNDANCE, energy.ACTIVITY} <= k)
names = sql(w, f"SELECT uniprot_id, gene_name FROM {CATALOG}.gold.dim_protein").set_index("uniprot_id").gene_name
print(f"训练集里的蛋白: {len(kinds)}，其中两种实验都做过的: {len(paired)}")
print("  ", ", ".join(sorted(names.get(p, p) for p in paired)))
print(labels.reads.value_counts().rename("标注的突变数").to_string())

### 结果怎么看
349 个训练蛋白中，**只有 9 个**同时有两类实验。其余 340 个对拆分没有信息。

### 训练数据的选择
只用两类数据训练：
- **9 个成对蛋白**：提供拆分所需的信息
- **14a 的 kcal/mol 数据**（218 个结构域）：只约束折叠能，并提供真实单位

其余 340 个蛋白不用。两种选择的实测对比：

| 训练数据 | KRAS 折叠预测 | 两个能量能否分开 |
|---|---|---|
| 全部 349 个蛋白 | 0.430 | 不能（两者与结合的相关：0.286 vs 0.325） |
| 9 个成对蛋白 + kcal/mol 数据 | 0.368 | 能（第 4 步 7 组全对） |

选后者。这一章的目标是把两个能量分开，不是单项精度。

### 面试怎么说
> "Two energies are not identifiable from an assay that sees only their product: any split giving the same readout fits equally well. I measured that — an L1 penalty collapses the functional energy, pinning barely helps — and solved it by training only on data that can tell them apart: the nine proteins measured both ways, plus the kcal/mol folding data as the ruler. It costs a little single-task accuracy."

---

## 第 3 步 · 训练

### 在干什么
用第 2 步选定的数据训练模型。

### 模型结构
- 输入：每个突变位置的 ESM-2 向量（1,280 维，蛋白语言模型对这个位置的表示）+ 结构特征
- 网络：LayerNorm → 512 → 128，和 08a、14a 相同
- 两个输出头：ΔΔG_fold 和 ΔΔG_func。每个头 20 个输出，对应 20 种氨基酸，突变成哪种取哪个
- 最后接第 1 步的换算器

### 训练设置
- 去掉和测试蛋白同源的结构域，防止泄漏
- 加权均方误差：每组实验权重相同，防止大实验主导；kcal/mol 数据占固定权重，用来固定能量的单位
- 3 个随机种子取平均

### 执行
约 2 分钟。输出依次是：去掉了 52 个同源结构域；103.5 万条测量中，32 万条用于训练（227 个蛋白 = 218 个结构域 + 9 个成对蛋白）；训练完成。

In [ ]:
splits = stability.load_splits(WORK / "mega_splits.pkl")
mega = stability.add_features(
    stability.megascale(WORK / "Tsuboyama2023_Dataset2_Dataset3_20230416.csv", splits),
    pd.read_parquet(WORK / "structure.parquet"), pd.read_parquet(WORK / "residue_index.parquet"),
    np.load(WORK / "logprobs.npy"))
mega = mega[mega.split.isin(["train", "val"])]
split_table = pd.read_csv(ROOT / "dbt/seeds/protein_split.csv")                       # 14a 的考卷规矩照旧
sequences = pd.read_parquet(MODELS / "proteome_sequences.parquet").set_index("uniprot_id").sequence
locked = [sequences[u] for u in split_table.loc[split_table.split == "test", "uniprot_id"] if u in sequences.index]
domains = mega.groupby("domain").seq.first()
related = {d for d, seq in domains.items() if any(validation.related(seq, s) for s in locked)}
mega = mega[~mega.domain.isin(related)]
print(f"大规模数据：去掉 {len(related)} 个和考试蛋白同源的结构域，剩 {mega.domain.nunique()} 个")
rows, vectors = energy.stack(mega, np.load(WORK / "vectors.npy"), labels,
                             pd.read_parquet(MODELS / "train_embeddings.parquet"),
                             np.asarray(np.load(MODELS / "train_embeddings.npy", mmap_mode="r")), titles.to_dict())
readout = rows[(rows.kind == energy.DDG) | rows.protein.isin(paired)]
print(f"全部测量 {len(rows):,} 行；训练读数层用 {len(readout):,} 行（{readout.protein.nunique()} 个蛋白）")
models = [energy.fit(readout, vectors, energy.FEATURES, seed=s) for s in (0, 1, 2)]
print("训练完成：3 个种子")

---

## 第 4 步 · 测试：KRAS

### 在干什么
预测 KRAS 每个突变的两个能量，和实验测出的能量比较。

### 为什么用 KRAS
KRAS 是细胞信号通路里的开关蛋白，常见的致癌基因，也是药物靶点。

- **模型没见过**：KRAS 和它唯一的同源蛋白 HRAS 都在 07a 锁定的测试集里
- **有带单位的真实值**：Weng 等 2024（Nature）用 2.6 万次测量，推算出每个突变的折叠能变化，以及和 6 个结合伙伴的结合能变化（RAF1、SOS1、PIK3CG、RALGDS，加上两个人工结合蛋白 DARPin K55、K27），单位 kcal/mol
- **能检验拆分**：如果拆分成立，预测的折叠能应该和实测折叠更相关，预测的功能能应该和实测结合更相关。一共 7 组比较

### 评价方法
Spearman 相关：两列数的排序是否一致，1 为完全一致，0 为无关。

实验数据的符号是"正 = 破坏"，我们是"正 = 稳定"，下面已经统一翻转，**越大越好**。

### 执行
第一格准备 KRAS 的特征并预测；第二格逐组比较。

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "databricks" / "src"))
from lakehouse import language_model as lm
URNS = {"urn:mavedb:00000115-a-7": "folding", "urn:mavedb:00000115-a-6": "binding_RAF1",
        "urn:mavedb:00000115-a-3": "binding_SOS1", "urn:mavedb:00000115-a-4": "binding_PIK3CG",
        "urn:mavedb:00000115-a-5": "binding_RALGDS", "urn:mavedb:00000115-a-1": "binding_DARPinK55",
        "urn:mavedb:00000115-a-2": "binding_DARPinK27"}
if not (KRAS / "kras_vectors.npy").exists():
    urns = ",".join(f"'{u}'" for u in URNS)
    query = ("SELECT urn, uniprot_position AS position, ref_aa, alt_aa, score "
             f"FROM {CATALOG}.silver.mavedb_variants "
             f"WHERE urn IN ({urns}) AND kind = 'missense' AND ref_matches_uniprot")
    sql(w, query).assign(measurement=lambda t: t.urn.map(URNS)).astype(
        {"position": int, "score": float}).to_parquet(KRAS / "kras_energies.parquet")
    sequence = sql(w, f"SELECT sequence FROM {CATALOG}.gold.dim_protein WHERE uniprot_id = 'P01116'").sequence[0]
    residues = pd.read_parquet(MODELS / "proteome_residues.parquet",
                               filters=[("uniprot_id", "==", "P01116")]).sort_values("position")
    model, tokenizer = lm.load()                                  # 和 07e、14a 同一个钉死的版本
    logprobs, vectors = lm.read_proteins([sequence], model, tokenizer, embeddings=True)[0]
    print("和平台算的对数概率最大差:", np.abs(np.stack(residues.logprobs.values) - logprobs).max())
    np.save(KRAS / "kras_vectors.npy", vectors.astype(np.float16))
    np.save(KRAS / "kras_logprobs.npy", logprobs)
    pd.DataFrame({"domain": "P01116", "position": np.arange(1, len(sequence) + 1)}).to_parquet(KRAS / "kras_index.parquet")
    residues[["uniprot_id", "position", "aa", "plddt", "rsa", "contacts_8", "contacts_12", "hse_up", "hse_down"]
             ].rename(columns={"uniprot_id": "domain"}).to_parquet(KRAS / "kras_structure.parquet")

measured = pd.read_parquet(KRAS / "kras_energies.parquet")
wide = measured.pivot_table(index=["position", "ref_aa", "alt_aa"], columns="measurement", values="score").reset_index()
kras = stability.add_features(
    wide.assign(domain="P01116", wt_aa=wide.ref_aa, mut_aa=wide.alt_aa, ddg=0.0),
    pd.read_parquet(KRAS / "kras_structure.parquet"), pd.read_parquet(KRAS / "kras_index.parquet"),
    np.load(KRAS / "kras_logprobs.npy"), strict=False)
kras["row"] += len(vectors)
every = np.concatenate([vectors, np.load(KRAS / "kras_vectors.npy")])
fold, func = zip(*(energy.energies(m, s, kras, every, energy.FEATURES) for m, s, _ in models))
kras["predicted_fold"], kras["predicted_func"] = np.mean(fold, axis=0), np.mean(func, axis=0)
print(f"KRAS 突变数 {len(kras):,}，位置 {kras.position.min()}–{kras.position.max()}")
kras[["position", "ref_aa", "alt_aa", "folding", "binding_RAF1", "predicted_fold", "predicted_func"]].head(3).round(3)

In [ ]:
rho = lambda a, b: -stats.spearmanr(a, b).statistic          # 实验是"正 = 破坏"，翻过来
rng = np.random.default_rng(0)
rows_out = []
for column in ["folding", "binding_RAF1", "binding_SOS1", "binding_PIK3CG", "binding_RALGDS",
               "binding_DARPinK55", "binding_DARPinK27"]:
    g = kras.dropna(subset=[column])
    wins = sum((lambda t: (rho(t[column], t.predicted_fold) > rho(t[column], t.predicted_func))
                          == (column == "folding"))(g.iloc[rng.integers(0, len(g), len(g))]) for _ in range(1000))
    rows_out.append({"实验测的": column, "突变数": len(g),
                     "我们的折叠能": rho(g[column], g.predicted_fold),
                     "我们的功能能": rho(g[column], g.predicted_func),
                     "ESM-2 单分数": rho(g[column], g.esm_delta),
                     "埋藏 × 保守性": rho(g[column], g.rsalor),
                     "正确的能量胜出": wins / 1000})
pd.DataFrame(rows_out).round(3)

### 结果怎么看

表里每个数都是**相关系数**：左边那一组实验的真实值，和上面那个预测值，在 3,076 个 KRAS 突变上的 Spearman 相关。

| 实验测的真实值 | 和预测折叠能的相关 | 和预测功能能的相关 | 和 ESM-2 单分数的相关 |
|---|---|---|---|
| **折叠** | **0.368** | 0.245 | 0.366 |
| 结合 RAF1 | 0.220 | **0.310** | 0.359 |
| 结合 SOS1 | 0.187 | **0.279** | 0.366 |
| 结合 PIK3CG | 0.173 | **0.314** | 0.326 |
| 结合 RALGDS | 0.221 | **0.388** | 0.386 |
| 结合 DARPin K55 | 0.268 | **0.406** | 0.433 |
| 结合 DARPin K27 | 0.155 | **0.223** | 0.286 |

- 实测折叠：和预测折叠能更相关（0.368 > 0.245）
- 6 组实测结合：都和预测功能能更相关（中位数 0.312 vs 0.204）
- **7 组方向全部正确。** 最后一列是按突变重抽 1000 次、对的能量赢的比例，全是 1.0。按突变重抽会高估确定性，更严格的按位置重抽见下一节：6 组显著，1 组不显著

**和单一分数比较：** ESM-2 对折叠 0.366，对结合 0.362，几乎相同，区分不了两种机制。

**局限：** 绝对精度没有超过 ESM-2（结合 0.312 vs 0.362，折叠持平）。这一章的收益是机制信息，不是精度。

### 面试怎么说
> "On KRAS, held out together with its only relative, the predicted folding energy tracks the measured folding ΔΔG and the predicted functional energy tracks binding to all six partners — the right energy wins all seven comparisons, six of them significant when whole positions are resampled. A single ESM-2 score correlates 0.366 with folding and 0.362 with binding: same answer for both. We trade a little absolute accuracy for the mechanism."

### 统计显著性：按位置重抽

上面"正确的能量胜出"那一列是**按突变**重抽的：3,000 多个突变随机重抽 1000 次。但同一个位置上的 19 个突变效果很像，并不独立。按突变重抽等于把样本量当成约 3,000，实际独立的只有约 166 个位置，结论会显得过于确定。

更严格的做法是**按位置重抽**：每次抽整个位置（连同它的全部突变），重算"对的能量的相关 − 错的能量的相关"，重复 2000 次（`validation.bootstrap_gap`）。

这一格用的是本次运行重新训练的模型，相关系数和上表可能差 0.002 以内（GPU 上的训练不是逐位可复现的）。

### 执行

In [ ]:
measurements = ["folding", "binding_RAF1", "binding_SOS1", "binding_PIK3CG", "binding_RALGDS",
                "binding_DARPinK55", "binding_DARPinK27"]
significance = []
for column in measurements:
    g = kras.dropna(subset=[column]).assign(measured=lambda t: -t[column])    # 实验是"正 = 破坏"，翻成"正 = 稳定"
    right, wrong = (("predicted_fold", "predicted_func") if column == "folding"
                    else ("predicted_func", "predicted_fold"))
    by_mutation = validation.bootstrap_gap(g, "measured", right, wrong)
    by_position = validation.bootstrap_gap(g, "measured", right, wrong, group="position")
    significance.append({"实验测的": column, "位置数": g.position.nunique(), "差值（对 − 错）": by_position["gap"],
                         "按突变重抽：赢的比例": by_mutation["wins"], "按位置重抽：赢的比例": by_position["wins"],
                         "单侧 p（按位置）": by_position["p"],
                         "95% 区间（按位置）": f"{by_position['low']:.3f} – {by_position['high']:.3f}"})
pd.DataFrame(significance).round(3)

### 结果怎么看
- **按突变重抽**：7 组全是 1.0，和上表一致。
- **按位置重抽**：7 组方向全对，**6 组在 0.05 水平显著**（折叠、RALGDS、DARPin K55、PIK3CG 的 p ≤ 0.003；RAF1、SOS1 的 p 约 0.015）。**DARPin K27 不显著**（p = 0.06，区间跨过 0）。

所以准确的说法是"**7 组方向全对，按位置重抽 6 组显著**"，而不是"1000 次 100%"。

另外，6 组结合来自同一个蛋白，彼此相关，不是 6 个独立的检验。所以下面再加两个蛋白。

### 再加两个没见过的蛋白：CYP2C9、G6PD

KRAS 只有一个蛋白。锁死的测试集里还有两个蛋白**两种实验都做过**：

- **CYP2C9**（肝脏里代谢药物的酶）：VAMP-seq 测蛋白量，Click-seq 测活性
- **G6PD**（红细胞里的代谢酶）：VAMP-seq 测蛋白量，另有一组综合功能评分

它们没有 KRAS 那种推算好的能量，真实值这样定：

- **稳定性的真实值** = 蛋白量
- **功能的真实值** = 活性里蛋白量解释不了的部分。先用蛋白量拟合活性（保序回归：只要求蛋白量越高、活性不降低），剩下的残差就是功能（`sites.beyond_abundance`）

检验和 KRAS 一样：蛋白量应该和预测折叠能更相关，残差应该和预测功能能更相关；按位置重抽算显著性。

### 执行

In [ ]:
from qafi_core import sites
PAIRED_TEST = {"CYP2C9": ("CYP2C9 abundance scores as measured by VAMP-seq",
                          "CYP2C9 activity scores as measured by Click-seq"),
               "G6PD": ("G6PD VAMP-seq Scores", "G6PD Combined Functional Scores")}
test = pd.read_parquet(MODELS / "test_variants.parquet")
test = test[test.dataset_id.map(titles).isin([t for pair in PAIRED_TEST.values() for t in pair])].reset_index(drop=True)
test_rows, test_vectors = energy.stack(mega.iloc[:0], vectors[:0], test, pd.read_parquet(MODELS / "test_embeddings.parquet"),
                                       np.asarray(np.load(MODELS / "test_embeddings.npy", mmap_mode="r")), titles.to_dict())
fold, func = zip(*(energy.energies(m, s, test_rows, test_vectors, energy.FEATURES) for m, s, _ in models))
test = test.assign(title=test.dataset_id.map(titles), predicted_fold=np.mean(fold, axis=0),
                   predicted_func=np.mean(func, axis=0))
held_out = []
for gene, (abundance, activity) in PAIRED_TEST.items():
    keys = ["uniprot_position", "alt_aa"]
    both = test[test.title == abundance].merge(test[test.title == activity][keys + ["label"]], on=keys,
                                               suffixes=("_abundance", "_activity"))
    both["function"] = sites.beyond_abundance(both.label_abundance, both.label_activity)
    for what, measured, right, wrong in [("蛋白量 → 应该像折叠能", "label_abundance", "predicted_fold", "predicted_func"),
                                         ("活性去掉蛋白量 → 应该像功能能", "function", "predicted_func", "predicted_fold")]:
        out = validation.bootstrap_gap(both, measured, right, wrong, group="uniprot_position")
        held_out.append({"蛋白": gene, "检验": what, "突变数": len(both), "位置数": both.uniprot_position.nunique(),
                         "和对的能量的相关": stats.spearmanr(both[measured], both[right])[0],
                         "和错的能量的相关": stats.spearmanr(both[measured], both[wrong])[0],
                         "按位置重抽：赢的比例": out["wins"], "单侧 p": out["p"]})
pd.DataFrame(held_out).round(3)

### 结果怎么看

| 蛋白 | 稳定性一侧（蛋白量） | 功能一侧（活性去掉蛋白量） |
|---|---|---|
| **CYP2C9** | 和折叠能更相关：0.618 vs 0.454，按位置重抽 100% ✓ | 0.253 vs 0.237，只赢 72%，**不显著** |
| **G6PD** | 0.520 vs 0.367，100% ✓ | 0.562 vs 0.400，100% ✓ |

- 稳定性一侧，两个蛋白都清楚。
- 功能一侧，G6PD 清楚，CYP2C9 分不开。CYP2C9 的功能真实值是从两组读数推出来的残差，噪声大；也可能模型在这个蛋白上确实没分开。两种可能都排除不了，如实记下。

**三个没见过的蛋白合起来**：稳定性一侧 3 个都对、都显著；功能一侧 KRAS 和 G6PD 显著，CYP2C9 不显著。

### 面试怎么说
> "I tested significance by resampling whole positions rather than mutations, because the 19 substitutions at a position are not independent. On KRAS the right energy still wins all seven comparisons, six of them at p < 0.05. Two more held-out proteins measured both ways, CYP2C9 and G6PD, reproduce the folding side cleanly; the functional side holds on G6PD but not on CYP2C9."

### 和已发表模型比较：AlphaMissense、CPT-1

AlphaMissense（DeepMind 2023）和 CPT-1 都对每个突变输出一个致病性分数。用同样的 7 组实测能量计算它们的相关，看单一分数能不能区分折叠和结合。

### 执行

In [ ]:
from qafi_platform.lakehouse import sql
published = sql(w, "SELECT position, ref_aa, alt_aa, am_pathogenicity, cpt1 "
                   f"FROM {CATALOG}.silver.variant_scores WHERE uniprot_id = 'P01116'"
                ).astype({"position": int, "am_pathogenicity": float, "cpt1": float})
both = kras.astype({"position": int}).merge(published, on=["position", "ref_aa", "alt_aa"])
compare = []
for name, column, flip in [("我们的折叠能", "predicted_fold", True), ("我们的功能能", "predicted_func", True),
                           ("AlphaMissense", "am_pathogenicity", False), ("CPT-1", "cpt1", False),
                           ("ESM-2 对数概率差", "esm_delta", True)]:
    sign = -1 if flip else 1
    scored = {m: sign * stats.spearmanr(g[m], g[column]).statistic
              for m in ["folding", "binding_RAF1", "binding_SOS1", "binding_PIK3CG", "binding_RALGDS",
                        "binding_DARPinK55", "binding_DARPinK27"]
              for g in [both.dropna(subset=[m, column])]}
    binding = np.median([v for k, v in scored.items() if k != "folding"])
    compare.append({"预测器": name, "对折叠": scored["folding"], "对结合（中位）": binding,
                    "差值（折叠 − 结合）": scored["folding"] - binding})
print(f"{len(both):,} 个 KRAS 突变同时有我们的能量和已发表的分数")
pd.DataFrame(compare).round(3)

### 结果怎么看

| 预测器 | 与折叠的相关 | 与结合的相关（中位） | 差值（折叠 − 结合） |
|---|---|---|---|
| **预测折叠能** | **0.368** | 0.204 | **+0.164** |
| **预测功能能** | 0.245 | **0.312** | **−0.067** |
| AlphaMissense | 0.293 | 0.382 | −0.089 |
| CPT-1 | 0.336 | 0.340 | −0.004 |
| ESM-2 | 0.366 | 0.362 | +0.004 |

看最后一列：
- ESM-2、CPT-1 差值约为 0：对折叠和结合没有区分
- AlphaMissense 是 −0.089，略偏结合；在结合上的精度高于我们的功能能（0.382 vs 0.312）
- 我们的两个能量是 +0.164 和 −0.067，**方向相反**

**结论：** 判断突变是否有害，AlphaMissense 更好；判断是稳定性问题还是功能问题，只有两个能量能回答。

### 面试怎么说
> "On the same KRAS measurements, AlphaMissense correlates 0.29 with folding and 0.38 with binding, ESM-2 gives 0.366 and 0.362 — essentially the same answer to both questions. Our two energies differ by 0.23 in which measurement they prefer, in opposite directions. AlphaMissense is the better score if the question is pathogenicity; it cannot answer which mechanism is broken, and that is the question that decides the next experiment."

---

## 第 5 步 · 能量可加

### 在干什么
验证两个突变的能量可以直接相加：只用单突变数据预测双突变。

### 为什么重要
蛋白工程要组合多个突变，组合数量太多，不可能全部测量。能量是可加的物理量；实验读数有上下限（会饱和），不能直接相加。

### 数据
14b 生成的 PSD95-PDZ3 数据：同一个文库同时测了蛋白量和结合。1,223 个单突变两类都有，另有约 5,000 个双突变。

### 方法（不使用任何双突变信息）
1. 用换算器反算：单突变的蛋白量读数 → ΔΔG_fold；结合读数减去折叠部分 → ΔΔG_func
2. 双突变的能量 = 两个单突变的能量之和
3. 用换算器正算，得到双突变的预测读数
4. 和实测的双突变比较

换算需要的三个量都不来自双突变：死亡水平（无义突变的读数）、野生型 = 0、野生型折叠能（单突变确定不了，在 0.4–3.0 之间扫描）。

对照方法：读数直接相加；取两者中较差的那个。

这一步不使用第 3 步训练的模型，只检验能量可加性。

### 执行

In [ ]:
RT = energy.RT
DMS = ROOT / "data/dms/results"
def read(kind):
    folder = DMS / f"{kind}/fitness/psd95_pdz3_{kind}"
    s = pd.read_csv(folder / "fitness_singles.txt", sep=r"\s+")
    d = pd.read_csv(folder / "fitness_doubles.txt", sep=r"\s+")
    n = len(pd.read_csv(folder / "fitness_wildtype.txt", sep=r"\s+").aa_seq.iloc[0])
    dead = float(s[s.STOP & (s.Pos < 0.9 * n)].fitness.median())   # 末端的无义突变不算，蛋白还能用
    s = s[~s.STOP & ~s.STOP_readthrough].assign(change=lambda t: t.Pos.astype(str) + t.Mut).set_index("change")
    d = d[~d.STOP & ~d.STOP_readthrough].assign(first=lambda t: t.Pos1.astype(str) + t.Mut1,
                                                second=lambda t: t.Pos2.astype(str) + t.Mut2,
                                                fitness=lambda t: t.fitness_uncorr)
    return {"singles": s, "doubles": d, "dead": dead}
data = {k: read(k) for k in ("abundance", "binding")}
shared = data["abundance"]["singles"].index.intersection(data["binding"]["singles"].index)
print(f"两种实验都测过的单突变 {len(shared)} 个 | 死亡水平 "
      f"蛋白量 {data['abundance']['dead']:.3f}、结合 {data['binding']['dead']:.3f}")

In [ ]:
def predict_doubles(g_fold_wt, g_func_wt):
    obs = {k: data[k]["singles"].loc[shared, "fitness"].to_numpy() for k in data}
    share_wt = {"abundance": energy.fraction_folded(g_fold_wt, RT),
                "binding": energy.fraction_functional(g_fold_wt, g_func_wt, RT)}
    scale = {k: -data[k]["dead"] / share_wt[k] for k in data}          # 野生型读数 = 0
    share = {k: np.clip((obs[k] - data[k]["dead"]) / scale[k], 1e-4, 1 - 1e-4) for k in data}
    g_fold = RT * np.log(share["abundance"] / (1 - share["abundance"]))     # 蛋白量 → 折叠能
    q = (1 - share["binding"]) / share["binding"]
    g_func = -RT * np.log(np.clip(q / (1 + np.exp(-g_fold / RT)), 1e-9, None))   # 结合里剩下的 → 功能能
    single = pd.DataFrame({"fold": g_fold - g_fold_wt, "func": g_func - g_func_wt}, index=shared)
    out = []
    for kind in data:
        d = data[kind]["doubles"]
        use = d[d["first"].isin(single.index) & d["second"].isin(single.index)]
        a, b = single.loc[use["first"]].to_numpy(), single.loc[use["second"]].to_numpy()
        total_fold, total_func = g_fold_wt + a[:, 0] + b[:, 0], g_func_wt + a[:, 1] + b[:, 1]
        guess = data[kind]["dead"] + scale[kind] * (
            energy.fraction_folded(total_fold, RT) if kind == "abundance"
            else energy.fraction_functional(total_fold, total_func, RT))
        y = use.fitness.to_numpy()
        fa = data[kind]["singles"].loc[use["first"], "fitness"].to_numpy()
        fb = data[kind]["singles"].loc[use["second"], "fitness"].to_numpy()
        r2 = lambda p: 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        for how, p in [("能量相加", guess), ("适应度相加", fa + fb), ("取差的那个", np.minimum(fa, fb))]:
            out.append({"实验": kind, "怎么预测": how, "双突变数": len(use),
                        "Pearson": stats.pearsonr(y, p)[0], "R²": r2(p)})
    return pd.DataFrame(out)

pd.DataFrame(predict_doubles(0.8, 1.0)).round(3)

### 结果怎么看

| 实验 | 方法 | 双突变数 | Pearson | R² |
|---|---|---|---|---|
| 蛋白量 | **能量相加** | 4,891 | **0.889** | **0.742** |
| 蛋白量 | 读数相加 | 4,891 | 0.822 | −0.041 |
| 蛋白量 | 取较差者 | 4,891 | 0.871 | 0.675 |
| 结合 | **能量相加** | 5,629 | **0.923** | **0.840** |
| 结合 | 读数相加 | 5,629 | 0.837 | −0.127 |
| 结合 | 取较差者 | 5,629 | 0.892 | 0.711 |

Pearson 衡量趋势，R² 衡量数值本身。R² < 0 表示比直接预测平均值还差。

- **能量相加**：R² 0.74 / 0.84
- **读数相加**：R² 为负。读数会饱和，相加后超出实际范围
- **取较差者**：0.68 / 0.71。它是物理模型在饱和区的近似

下面检查野生型折叠能取值的影响。

In [ ]:
scan = pd.concat([predict_doubles(f, 1.0).assign(野生型稳定性=f) for f in (0.4, 0.8, 1.2, 2.0, 3.0)])
scan[scan["怎么预测"] == "能量相加"].pivot_table(index="野生型稳定性", columns="实验", values="R²").round(3)

### 结果怎么看
野生型折叠能在 0.4–1.2 之间，R² 基本不变（0.74 / 0.84）；到 3.0 降到 0.53 / 0.63。在整个范围内，能量相加都远好于读数相加，结论不依赖这个取值。

### 说明
- 这一步需要同一文库的蛋白量、结合和双突变数据，只有 14b 生成了这种数据。
- PSD95-PDZ3 在训练集里（来自 Domainome），但这一步没有用训练好的模型，所以检验的是能量可加性本身。对没见过的蛋白的测试是第 4 步。

### 面试怎么说
> "Free energies add, fitness does not. Using only single mutants to fix each substitution's two energies, the thermodynamic model predicts ~5,000 held-out double mutants at R² 0.74 and 0.84, while adding the measured single fitness gives a negative R² because fitness saturates. The 'worse of the two' heuristic gets 0.68-0.71 — it is the saturating limit of the same physics."

---

## 第 6 步 · 记录

把参数和结果存进 MLflow（Databricks 的实验记录）。

In [ ]:
summary = pd.DataFrame(rows_out)
binding = summary[summary["实验测的"] != "folding"]
metrics = {
    "kras_folding_fold_energy": summary.iloc[0]["我们的折叠能"],
    "kras_folding_func_energy": summary.iloc[0]["我们的功能能"],
    "kras_binding_fold_energy": binding["我们的折叠能"].median(),
    "kras_binding_func_energy": binding["我们的功能能"].median(),
    "kras_right_energy_wins": summary["正确的能量胜出"].min(),
    "doubles_r2_abundance": 0.742, "doubles_r2_binding": 0.840,
}
tracking.experiment_run(
    EXPERIMENT, "two_energies",
    params={**energy.SETTINGS, "seeds": 3, "readout_proteins": len(paired),
            "readout_rows": len(readout), "all_rows": len(rows), "test": "KRAS (locked side)"},
    metrics={k: float(v) for k, v in metrics.items()},
    tables={"kras": summary},
    chapter="14c")
print("已记录"); pd.Series(metrics).round(3)

第 4 步补的两项检验（按位置重抽、CYP2C9 和 G6PD）另记一笔：

In [ ]:
tracking.experiment_run(
    EXPERIMENT, "two_energies_checks",
    params={"bootstrap": "by position, 2000 resamples", "held_out_paired": "CYP2C9, G6PD",
            "function_truth": "activity minus an isotonic fit on abundance"},
    metrics={**{f"kras_{r['实验测的']}_wins_by_position": float(r["按位置重抽：赢的比例"]) for r in significance},
             **{f"{r['蛋白']}_{'fold' if r['检验'].startswith('蛋白量') else 'func'}_wins_by_position":
                float(r["按位置重抽：赢的比例"]) for r in held_out}},
    tables={"kras_by_position": pd.DataFrame(significance), "held_out_paired": pd.DataFrame(held_out)},
    chapter="14c")
print("已记录")

---

## 这一章做完了什么

1. **换算器**：把两个能量换算成实验读数。测蛋白量的实验只用折叠能，两个输出因此分别对应稳定性和功能。
2. **不可辨识**：只有一类实验的蛋白，两个能量分不开。所以训练只用 9 个成对蛋白和 14a 的 kcal/mol 数据。
3. **KRAS 测试**（模型没见过）：预测折叠能对应实测折叠，预测功能能对应全部 6 组实测结合，7 组方向全对；按位置重抽，6 组显著。ESM-2 单分数对两者的相关几乎相同。
   再加两个没见过、两种实验都做过的蛋白：稳定性一侧 CYP2C9、G6PD 都对；功能一侧 G6PD 对，CYP2C9 不显著。
4. **能量可加**：只用单突变预测约 5,000 个双突变，R² 0.74 / 0.84；读数直接相加，R² 为负。

局限：单项精度没有超过最好的单分数模型。

## 下一步

**14d · 把两个能量变成一张位点地图**：用这一章的两个能量给每个位置贴标签（结构关键 / 功能关键 / 可以改），找出功能敏感、但稳定性不敏感的位置，检查它们是否落在已成药的口袋上。

（附录 **14x** 另外回答一个部署问题：一个新项目要测多少个突变才值得。它不用这一章的两个能量，所以不在这条线上。）

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14c two energies with a thermodynamic readout, tested on KRAS' && git log --oneline -1")